In [2]:
import pandas as pd
import numpy as np
import itertools
import pymc as pm
import arviz as az
import time

import sys
sys.path.append('../Source')
from Pipeline.build_country_comparisons import buildCountryComparisons

alignedCountryData = pd.read_csv('../Data/Processed/alignedDataCountry.csv')
countryComparisons = buildCountryComparisons(alignedCountryData)

caucasusData = countryComparisons[countryComparisons['region'] == 'Caucasus and Central Asia']
regionCountries = sorted(set(caucasusData['countryI']) | set(caucasusData['countryJ']))
regionIndexMap = {country: i for i, country in enumerate(regionCountries)}
nRegionCountries = len(regionCountries)
caucasusWeeks = sorted(caucasusData['week'].unique())

print(nRegionCountries, len(caucasusWeeks))

c:\Users\rcuhl\miniconda3\envs\georisk\Lib\site-packages\arviz\__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


9 391


In [2]:
initialWeekData = caucasusData[caucasusData['week'] == 0]

initialIndexI = caucasusData['countryI'].map(regionIndexMap).to_numpy()
initialIndexJ = caucasusData['countryJ'].map(regionIndexMap).to_numpy()
initialWinsI = caucasusData['winsI'].to_numpy()
initialTotals = caucasusData['total'].to_numpy()

with pm.Model() as rollingModel:
    indexIData = pm.Data('indexIData', initialIndexI)
    indexJData = pm.Data('indexJData', initialIndexJ)
    winsIData = pm.Data('winsIData', initialWinsI)
    totalsData = pm.Data('totalsData', initialTotals)

    beta = pm.ZeroSumNormal('beta', sigma=1.0, shape=nRegionCountries)
    probability = pm.math.sigmoid(beta[indexIData] - beta[indexJData])
    obs = pm.Binomial('obs', n=totalsData, p=probability, observed=winsIData)

In [3]:
weeklyBetaResults = []
weeksSkipped = []

startTime = time.time()

for week in caucasusWeeks:
    weekData = caucasusData[caucasusData['week'] == week]

    if len(weekData) == 0:
        weeksSkipped.append(week)
        continue

    indexI = weekData['countryI'].map(regionIndexMap).to_numpy()
    indexJ = weekData['countryJ'].map(regionIndexMap).to_numpy()
    winsI = weekData['winsI'].to_numpy()
    totals = weekData['total'].to_numpy()

    with rollingModel:
        pm.set_data({
            'indexIData': indexI,
            'indexJData': indexJ,
            'winsIData': winsI,
            'totalsData': totals
        })

        approx = pm.fit(n=20000, method='advi', progressbar=False)
        trace = approx.sample(400)

    betaSamples = trace.posterior['beta'].values.reshape(-1, nRegionCountries)
    betaMeans = betaSamples.mean(axis=0)
    betaStds = betaSamples.std(axis=0)

    for i, country in enumerate(regionCountries):
        weeklyBetaResults.append({
            'week': week,
            'region': 'Caucasus and Central Asia',
            'country': country,
            'betaMean': betaMeans[i],
            'betaStd': betaStds[i]
        })

endTime = time.time()

print(f"Total time: {endTime - startTime:.1f} seconds")
print(f"Weeks fit: {len(caucasusWeeks) - len(weeksSkipped)}")
print(f"Weeks skipped: {len(weeksSkipped)}")

Finished [100%]: Average Loss = 48.594
Finished [100%]: Average Loss = 61.498
Finished [100%]: Average Loss = 48.743
Finished [100%]: Average Loss = 61.644
Finished [100%]: Average Loss = 60.794
Finished [100%]: Average Loss = 63.605
Finished [100%]: Average Loss = 49.471
Finished [100%]: Average Loss = 50.007
Finished [100%]: Average Loss = 50.103
Finished [100%]: Average Loss = 72.581
Finished [100%]: Average Loss = 60.64
Finished [100%]: Average Loss = 60.138
Finished [100%]: Average Loss = 59.489
Finished [100%]: Average Loss = 75.164
Finished [100%]: Average Loss = 63.87
Finished [100%]: Average Loss = 51.686
Finished [100%]: Average Loss = 64.344
Finished [100%]: Average Loss = 73.337
Finished [100%]: Average Loss = 65.258
Finished [100%]: Average Loss = 51.385
Finished [100%]: Average Loss = 61.115
Finished [100%]: Average Loss = 63.973
Finished [100%]: Average Loss = 76.455
Finished [100%]: Average Loss = 60.533
Finished [100%]: Average Loss = 50.795
Finished [100%]: Average Lo

Total time: 1159.3 seconds
Weeks fit: 391
Weeks skipped: 0


In [4]:
weeklyBetaCaucasusDF = pd.DataFrame(weeklyBetaResults)
print(weeklyBetaCaucasusDF.shape)
weeklyBetaCaucasusDF.head(10)

(3519, 5)


,week,region,country,betaMean,betaStd
0,2018-01-01/2018-01-07,Caucasus and Central Asia,Afghanistan,2.221638,0.163267
1,2018-01-01/2018-01-07,Caucasus and Central Asia,Armenia,-0.332028,0.222841
2,2018-01-01/2018-01-07,Caucasus and Central Asia,Azerbaijan,2.187950,0.164607
3,2018-01-01/2018-01-07,Caucasus and Central Asia,Georgia,-0.234997,0.237734
4,2018-01-01/2018-01-07,Caucasus and Central Asia,Kazakhstan,-1.113216,0.284975
5,2018-01-01/2018-01-07,Caucasus and Central Asia,Kyrgyzstan,-1.978129,0.390205
6,2018-01-01/2018-01-07,Caucasus and Central Asia,Tajikistan,0.560879,0.885285
7,2018-01-01/2018-01-07,Caucasus and Central Asia,Turkmenistan,0.620993,0.831263
8,2018-01-01/2018-01-07,Caucasus and Central Asia,Uzbekistan,-1.933090,0.471682
9,2018-01-08/2018-01-14,Caucasus and Central Asia,Afghanistan,2.404007,0.127639


In [5]:
weeklyBetaCaucasusDF.to_csv('../Data/Processed/weeklyBetaCaucasusCA.csv', index=False)

In [6]:
middleEastData = countryComparisons[countryComparisons['region'] == 'Middle East']
middleEastCountries = sorted(set(middleEastData['countryI']) | set(middleEastData['countryJ']))
middleEastIndexMap = {country: i for i, country in enumerate(middleEastCountries)}
nMiddleEastCountries = len(middleEastCountries)
middleEastWeeks = sorted(middleEastData['week'].unique())

print(nMiddleEastCountries, len(middleEastWeeks))

13 391


In [7]:
initialMEWeekData = middleEastData[middleEastData['week'] == middleEastWeeks[0]]

initialMEIndexI = initialMEWeekData['countryI'].map(middleEastIndexMap).to_numpy()
initialMEIndexJ = initialMEWeekData['countryJ'].map(middleEastIndexMap).to_numpy()
initialMEWinsI = initialMEWeekData['winsI'].to_numpy()
initialMETotals = initialMEWeekData['total'].to_numpy()

with pm.Model() as middleEastRollingModel:
    indexIData = pm.Data('indexIData', initialMEIndexI)
    indexJData = pm.Data('indexJData', initialMEIndexJ)
    winsIData = pm.Data('winsIData', initialMEWinsI)
    totalsData = pm.Data('totalsData', initialMETotals)

    beta = pm.ZeroSumNormal('beta', sigma=1.0, shape=nMiddleEastCountries)
    p = pm.math.sigmoid(beta[indexIData] - beta[indexJData])
    obs = pm.Binomial('obs', n=totalsData, p=p, observed=winsIData)

In [8]:
weeklyBetaResultsME = []
weeksSkippedME = []

startTime = time.time()

for week in middleEastWeeks:
    weekData = middleEastData[middleEastData['week'] == week]

    if len(weekData) == 0:
        weeksSkippedME.append(week)
        continue

    indexI = weekData['countryI'].map(middleEastIndexMap).to_numpy()
    indexJ = weekData['countryJ'].map(middleEastIndexMap).to_numpy()
    winsI = weekData['winsI'].to_numpy()
    totals = weekData['total'].to_numpy()

    with middleEastRollingModel:
        pm.set_data({
            'indexIData': indexI,
            'indexJData': indexJ,
            'winsIData': winsI,
            'totalsData': totals
        })

        approx = pm.fit(n=20000, method='advi', progressbar=False)
        trace = approx.sample(400)

    betaSamples = trace.posterior['beta'].values.reshape(-1, nMiddleEastCountries)
    betaMeans = betaSamples.mean(axis=0)
    betaStds = betaSamples.std(axis=0)

    for i, country in enumerate(middleEastCountries):
        weeklyBetaResultsME.append({
            'week': week,
            'region': 'Middle East',
            'country': country,
            'betaMean': betaMeans[i],
            'betaStd': betaStds[i]
        })

endTime = time.time()

print(f"Total time: {endTime - startTime:.1f} seconds")
print(f"Weeks fit: {len(middleEastWeeks) - len(weeksSkippedME)}")
print(f"Weeks skipped: {len(weeksSkippedME)}")

Finished [100%]: Average Loss = 138.4
Finished [100%]: Average Loss = 168.16
Finished [100%]: Average Loss = 170.1
Finished [100%]: Average Loss = 171.09
Finished [100%]: Average Loss = 175.72
Finished [100%]: Average Loss = 174.99
Finished [100%]: Average Loss = 173.45
Finished [100%]: Average Loss = 169.88
Finished [100%]: Average Loss = 162.68
Finished [100%]: Average Loss = 170.66
Finished [100%]: Average Loss = 174.14
Finished [100%]: Average Loss = 167.44
Finished [100%]: Average Loss = 157.78
Finished [100%]: Average Loss = 178.45
Finished [100%]: Average Loss = 173.2
Finished [100%]: Average Loss = 168.9
Finished [100%]: Average Loss = 168.07
Finished [100%]: Average Loss = 165.48
Finished [100%]: Average Loss = 188.82
Finished [100%]: Average Loss = 197.27
Finished [100%]: Average Loss = 167.61
Finished [100%]: Average Loss = 171.83
Finished [100%]: Average Loss = 168.97
Finished [100%]: Average Loss = 162.22
Finished [100%]: Average Loss = 161.45
Finished [100%]: Average Loss

Total time: 1145.9 seconds
Weeks fit: 391
Weeks skipped: 0


In [9]:
weeklyBetaMEDF = pd.DataFrame(weeklyBetaResultsME)
print(weeklyBetaMEDF.shape)
weeklyBetaMEDF.head(13)

(5083, 5)


,week,region,country,betaMean,betaStd
0,2018-01-01/2018-01-07,Middle East,Bahrain,0.217842,0.101907
1,2018-01-01/2018-01-07,Middle East,Iran,1.165205,0.076633
2,2018-01-01/2018-01-07,Middle East,Iraq,-0.020646,0.100990
3,2018-01-01/2018-01-07,Middle East,Israel,-1.924097,0.176641
4,2018-01-01/2018-01-07,Middle East,Jordan,-1.898124,0.187803
5,2018-01-01/2018-01-07,Middle East,Kuwait,0.581055,0.360420
6,2018-01-01/2018-01-07,Middle East,Lebanon,-1.643885,0.161427
7,2018-01-01/2018-01-07,Middle East,Palestine,0.606012,0.094614
8,2018-01-01/2018-01-07,Middle East,Qatar,0.565518,0.371563
9,2018-01-01/2018-01-07,Middle East,Saudi Arabia,-0.963435,0.137866


In [10]:
weeklyBetaMEDF.to_csv('../Data/Processed/weeklyBetaMiddleEast.csv', index=False)

In [11]:
northAfricaData = countryComparisons[countryComparisons['region'] == 'Northern Africa']
northAfricaCountries = sorted(set(northAfricaData['countryI']) | set(northAfricaData['countryJ']))
northAfricaIndexMap = {country: i for i, country in enumerate(northAfricaCountries)}
nNorthAfricaCountries = len(northAfricaCountries)
northAfricaWeeks = sorted(northAfricaData['week'].unique())

print(nNorthAfricaCountries, len(northAfricaWeeks))

6 391


In [12]:
initialNAWeekData = northAfricaData[northAfricaData['week'] == northAfricaWeeks[0]]

initialNAIndexI = initialNAWeekData['countryI'].map(northAfricaIndexMap).to_numpy()
initialNAIndexJ = initialNAWeekData['countryJ'].map(northAfricaIndexMap).to_numpy()
initialNAWinsI = initialNAWeekData['winsI'].to_numpy()
initialNATotals = initialNAWeekData['total'].to_numpy()

with pm.Model() as northAfricaRollingModel:
    indexIData = pm.Data('indexIData', initialNAIndexI)
    indexJData = pm.Data('indexJData', initialNAIndexJ)
    winsIData = pm.Data('winsIData', initialNAWinsI)
    totalsData = pm.Data('totalsData', initialNATotals)

    beta = pm.ZeroSumNormal('beta', sigma=1.0, shape=nNorthAfricaCountries)
    p = pm.math.sigmoid(beta[indexIData] - beta[indexJData])
    obs = pm.Binomial('obs', n=totalsData, p=p, observed=winsIData)

In [13]:
weeklyBetaResultsNA = []
weeksSkippedNA = []

startTime = time.time()

for week in northAfricaWeeks:
    weekData = northAfricaData[northAfricaData['week'] == week]

    if len(weekData) == 0:
        weeksSkippedNA.append(week)
        continue

    indexI = weekData['countryI'].map(northAfricaIndexMap).to_numpy()
    indexJ = weekData['countryJ'].map(northAfricaIndexMap).to_numpy()
    winsI = weekData['winsI'].to_numpy()
    totals = weekData['total'].to_numpy()

    with northAfricaRollingModel:
        pm.set_data({
            'indexIData': indexI,
            'indexJData': indexJ,
            'winsIData': winsI,
            'totalsData': totals
        })

        approx = pm.fit(n=20000, method='advi', progressbar=False)
        trace = approx.sample(400)

    betaSamples = trace.posterior['beta'].values.reshape(-1, nNorthAfricaCountries)
    betaMeans = betaSamples.mean(axis=0)
    betaStds = betaSamples.std(axis=0)

    for i, country in enumerate(northAfricaCountries):
        weeklyBetaResultsNA.append({
            'week': week,
            'region': 'Northern Africa',
            'country': country,
            'betaMean': betaMeans[i],
            'betaStd': betaStds[i]
        })

endTime = time.time()

print(f"Total time: {endTime - startTime:.1f} seconds")
print(f"Weeks fit: {len(northAfricaWeeks) - len(weeksSkippedNA)}")
print(f"Weeks skipped: {len(weeksSkippedNA)}")

Finished [100%]: Average Loss = 32.238
Finished [100%]: Average Loss = 44.214
Finished [100%]: Average Loss = 38.191
Finished [100%]: Average Loss = 34.996
Finished [100%]: Average Loss = 35.888
Finished [100%]: Average Loss = 31.547
Finished [100%]: Average Loss = 34.73
Finished [100%]: Average Loss = 33.671
Finished [100%]: Average Loss = 32.441
Finished [100%]: Average Loss = 34.471
Finished [100%]: Average Loss = 33.043
Finished [100%]: Average Loss = 32.531
Finished [100%]: Average Loss = 23.199
Finished [100%]: Average Loss = 16.63
Finished [100%]: Average Loss = 29.66
Finished [100%]: Average Loss = 30.986
Finished [100%]: Average Loss = 32.293
Finished [100%]: Average Loss = 28.31
Finished [100%]: Average Loss = 29.241
Finished [100%]: Average Loss = 32.34
Finished [100%]: Average Loss = 34.133
Finished [100%]: Average Loss = 25.951
Finished [100%]: Average Loss = 31.766
Finished [100%]: Average Loss = 23.55
Finished [100%]: Average Loss = 28.863
Finished [100%]: Average Loss =

Total time: 990.3 seconds
Weeks fit: 391
Weeks skipped: 0


In [14]:
weeklyBetaNADF = pd.DataFrame(weeklyBetaResultsNA)
print(weeklyBetaNADF.shape)
weeklyBetaNADF.head(6)

(2346, 5)


,week,region,country,betaMean,betaStd
0,2018-01-01/2018-01-07,Northern Africa,Algeria,0.257720,0.212217
1,2018-01-01/2018-01-07,Northern Africa,Egypt,0.288021,0.207301
2,2018-01-01/2018-01-07,Northern Africa,Libya,0.270538,0.186070
3,2018-01-01/2018-01-07,Northern Africa,Morocco,-0.411485,0.230841
4,2018-01-01/2018-01-07,Northern Africa,Sudan,0.804794,0.184471
5,2018-01-01/2018-01-07,Northern Africa,Tunisia,-1.209588,0.199556


In [15]:
weeklyBetaNADF.to_csv('../Data/Processed/weeklyBetaNorthAfrica.csv', index=False)

In [3]:
europeData = countryComparisons[countryComparisons['region'] == 'Europe']
europeCountries = sorted(set(europeData['countryI']) | set(europeData['countryJ']))
europeIndexMap = {country: i for i, country in enumerate(europeCountries)}
nEuropeCountries = len(europeCountries)
europeWeeks = sorted(europeData['week'].unique())

print(nEuropeCountries, len(europeWeeks))

41 391


In [4]:
initialEuWeekData = europeData[europeData['week'] == europeWeeks[0]]

initialEuIndexI = initialEuWeekData['countryI'].map(europeIndexMap).to_numpy()
initialEuIndexJ = initialEuWeekData['countryJ'].map(europeIndexMap).to_numpy()
initialEuWinsI = initialEuWeekData['winsI'].to_numpy()
initialEuTotals = initialEuWeekData['total'].to_numpy()

with pm.Model() as europeRollingModel:
    indexIData = pm.Data('indexIData', initialEuIndexI)
    indexJData = pm.Data('indexJData', initialEuIndexJ)
    winsIData = pm.Data('winsIData', initialEuWinsI)
    totalsData = pm.Data('totalsData', initialEuTotals)

    beta = pm.ZeroSumNormal('beta', sigma=1.0, shape=nEuropeCountries)
    p = pm.math.sigmoid(beta[indexIData] - beta[indexJData])
    obs = pm.Binomial('obs', n=totalsData, p=p, observed=winsIData)

In [5]:
weeklyBetaResultsEurope = []
weeksSkippedEurope = []

startTime = time.time()

for week in europeWeeks:
    weekData = europeData[europeData['week'] == week]

    if len(weekData) == 0:
        weeksSkippedEurope.append(week)
        continue

    indexI = weekData['countryI'].map(europeIndexMap).to_numpy()
    indexJ = weekData['countryJ'].map(europeIndexMap).to_numpy()
    winsI = weekData['winsI'].to_numpy()
    totals = weekData['total'].to_numpy()

    with europeRollingModel:
        pm.set_data({
            'indexIData': indexI,
            'indexJData': indexJ,
            'winsIData': winsI,
            'totalsData': totals
        })

        approx = pm.fit(n=20000, method='advi', progressbar=False)
        trace = approx.sample(400)

    betaSamples = trace.posterior['beta'].values.reshape(-1, nEuropeCountries)
    betaMeans = betaSamples.mean(axis=0)
    betaStds = betaSamples.std(axis=0)

    for i, country in enumerate(europeCountries):
        weeklyBetaResultsEurope.append({
            'week': week,
            'region': 'Europe',
            'country': country,
            'betaMean': betaMeans[i],
            'betaStd': betaStds[i]
        })

endTime = time.time()

print(f"Total time: {endTime - startTime:.1f} seconds")
print(f"Weeks fit: {len(europeWeeks) - len(weeksSkippedEurope)}")
print(f"Weeks skipped: {len(weeksSkippedEurope)}")

Finished [100%]: Average Loss = 48.08
Finished [100%]: Average Loss = 100.17
Finished [100%]: Average Loss = 144.45
Finished [100%]: Average Loss = 134.46
Finished [100%]: Average Loss = 163.07
Finished [100%]: Average Loss = 151.26
Finished [100%]: Average Loss = 156.66
Finished [100%]: Average Loss = 164.41
Finished [100%]: Average Loss = 179
Finished [100%]: Average Loss = 177.06
Finished [100%]: Average Loss = 145.52
Finished [100%]: Average Loss = 138.04
Finished [100%]: Average Loss = 142.28
Finished [100%]: Average Loss = 168.7
Finished [100%]: Average Loss = 104.94
Finished [100%]: Average Loss = 154.79
Finished [100%]: Average Loss = 159.63
Finished [100%]: Average Loss = 147.46
Finished [100%]: Average Loss = 137.71
Finished [100%]: Average Loss = 158.55
Finished [100%]: Average Loss = 159.19
Finished [100%]: Average Loss = 132
Finished [100%]: Average Loss = 199.96
Finished [100%]: Average Loss = 198.23
Finished [100%]: Average Loss = 196.11
Finished [100%]: Average Loss = 1

Total time: 1069.3 seconds
Weeks fit: 391
Weeks skipped: 0


In [6]:
weeklyBetaEuropeDF = pd.DataFrame(weeklyBetaResultsEurope)
print(weeklyBetaEuropeDF.shape)
weeklyBetaEuropeDF.head(10)

(16031, 5)


,week,region,country,betaMean,betaStd
0,2018-01-01/2018-01-07,Europe,Albania,0.060609,1.077021
1,2018-01-01/2018-01-07,Europe,Austria,0.047717,1.024270
2,2018-01-01/2018-01-07,Europe,Belarus,-0.854684,0.394496
3,2018-01-01/2018-01-07,Europe,Belgium,0.111869,1.089182
4,2018-01-01/2018-01-07,Europe,Bosnia and Herzegovina,-0.013890,1.032200
5,2018-01-01/2018-01-07,Europe,Bulgaria,0.906500,0.255152
6,2018-01-01/2018-01-07,Europe,Croatia,0.065730,1.039528
7,2018-01-01/2018-01-07,Europe,Cyprus,0.087175,1.079257
8,2018-01-01/2018-01-07,Europe,Czech Republic,0.085987,1.057294
9,2018-01-01/2018-01-07,Europe,Denmark,0.031707,1.141569


In [7]:
weeklyBetaEuropeDF.to_csv('../Data/Processed/weeklyBetaEurope.csv', index=False)